# ArtifactContourCombined
Self-contained notebook: **Magi text boxes + Paddle OCR → local Gemma artifact classification → stage 1 filtering → stage 2 contour grouping and SPOKEN/NARRATED roles → tail override**.

Edit the sequence/path settings, then Run All in `godfather.venv`. Raw OCR and bubble caches are reused where compatible; missing data is created locally. Gemma runs in this notebook—no classification export from another notebook is required. Only Paddle text is sent to Gemma.

**Stage 1:** Remove Gemma Artifacts. Empty Paddle + punctuation-only Magi fallback requires a qualifying bubble mask, otherwise delete. No Artifact rescue.

**Stage 2:** Apply the existing individual outlines, faint retries, overlapping merge candidates, concatenation and white/solid-region role classification to survivors. Explicit Magi text-to-tail association always forces SPOKEN; any linked member forces its group SPOKEN. Original contour roles/reasons are retained. Character identity assignment is outside this experiment.

Two single-page viewers show before and after filtering/roles. Raw OCR, Gemma responses, deletions, grouping, role evidence and tail overrides are exported separately from the production pipeline.

In [55]:
import os
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'
import torch
import json,hashlib,math,gc,unicodedata,importlib.util,sys,re
from difflib import SequenceMatcher
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display,clear_output,Image as DisplayImage

PROJECT=Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
PIPELINE_ROOT=PROJECT/'paddleOCR-Magi'
SPLIT='development'
SEQUENCE_ID='seq_14469aeb38963085'
PAGE_NUMBER=None  # None = every exported page; integer = one page.
BUBBLE_MODEL=Path('C:/MODELS/manga_bubbles/bubbles_detect.pt')
BUBBLE_SETTINGS=dict(imgsz=1024,confidence=0.25)
FORCE_BUBBLES=False
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
CACHE=PIPELINE_ROOT/'cache_artifact_contour_combined'
CONTOUR_HELPER=PIPELINE_ROOT/'contour_narrator2.py'


def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=True,allow_nan=False).encode()).hexdigest()


def file_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_payload(path):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    assert digest(payload['data'])==payload['data_hash'],f'Invalid cache checksum: {path}'
    return payload['data']


def punctuation_only(text):
    text=str(text or '').strip()
    return bool(text) and all(c.isspace() or unicodedata.category(c).startswith('P') for c in text)

# All stages run here; no exported classifications are required from another notebook.
MAGI_MODEL=Path('C:/MODELS/magiv2')
GEMMA_MODEL=Path('C:/MODELS/gemma-3-4b-it')
PADDLE_DET_PATH=Path.home()/'.paddlex/official_models/PP-OCRv5_server_det'
PADDLE_REC_PATH=Path.home()/'.paddlex/official_models/en_PP-OCRv5_mobile_rec'
FORCE_GEMMA=False  # True reruns Gemma; False reuses this notebook's matching decision cache.
EXPERIMENT=PROJECT/'FINETUNING-EXPERMENTS/NarrationModelwBubbles'
CACHE_ROOTS=[PIPELINE_ROOT/'cache_contour_full_inference',
             EXPERIMENT/'cache_selected_ocr/full_pipeline',PIPELINE_ROOT/'cache_inference']


## 1. Select pages and discover OCR caches
Select the sequence above. Reuse raw Magi/Paddle OCR by page checksum; do not load another notebook's classification export. Missing OCR is generated locally below.

In [56]:
# Ignore whitespace accidentally pasted with sequence IDs or split names.
SEQUENCE_ID=SEQUENCE_ID.strip()
SPLIT=SPLIT.strip()
sequence_folder=PROJECT/'TaskData'/SPLIT/'images'/SEQUENCE_ID
if not sequence_folder.is_dir():
    matches=[p for p in (PROJECT/'TaskData').glob('*/images/'+SEQUENCE_ID) if p.is_dir()]
    if len(matches)==1:
        sequence_folder=matches[0]
        SPLIT=sequence_folder.parent.parent.name
        print('Found sequence in split:',SPLIT)
    else:
        raise FileNotFoundError(f'Sequence folder not found: {sequence_folder}. Check SEQUENCE_ID and PROJECT. Matches: {matches}')
PAGE_FILES=sorted(p for p in sequence_folder.iterdir() if p.suffix.lower()=='.png')
available_pages=[p.name for p in PAGE_FILES]
if PAGE_NUMBER is not None:
    PAGE_NUMBER=int(PAGE_NUMBER)
    PAGE_FILES=[p for p in PAGE_FILES if p.stem.isdigit() and int(p.stem)==PAGE_NUMBER]
if not PAGE_FILES:
    raise FileNotFoundError(f'No selected PNG pages in {sequence_folder}. PAGE_NUMBER={PAGE_NUMBER}; available pages={available_pages}')
CACHE.mkdir(parents=True,exist_ok=True)
def file_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=True,allow_nan=False).encode()).hexdigest()
def read_payload(path):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    assert digest(payload['data'])==payload['data_hash'],f'Invalid cache checksum: {path}'
    return payload['data']
print('Kernel:',sys.executable)
print('Pages:',[p.name for p in PAGE_FILES])

OUTPUT=CACHE/SPLIT/SEQUENCE_ID/('sequence' if PAGE_NUMBER is None else f'page_{PAGE_NUMBER:02d}')
OUTPUT.mkdir(parents=True,exist_ok=True)


Found sequence in split: test
Kernel: c:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Pages: ['01.png', '02.png', '03.png']


In [57]:
def cached_page(path):
    sha=file_hash(path)
    with Image.open(path) as im: size=im.size
    for root in CACHE_ROOTS:
        for manifest_path in sorted(root.rglob('manifest.json')):
            try:
                manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
                matches=[i for i,p in enumerate(manifest['inputs']['pages'])
                         if p['sha256']==sha and (p['width'],p['height'])==size]
                if len(matches)!=1:continue
                index=matches[0]
                selection_path=manifest_path.with_name('ocr_selection.json')
                if selection_path.is_file():
                    page=read_payload(selection_path)[index]
                    if all(v['text_index']==i for i,v in enumerate(page)):
                        return [dict(box=v['box'],magi_text=v['magi_text'],paddle_text=v['paddle_text'],
                            paddle_lines=v.get('paddle_lines',[]),selected_text=v.get('selected_text',''),
                            paddle_status=v.get('paddle_status','cached'),paddle_error=v.get('paddle_error')) for v in page],str(selection_path)
                raw=read_payload(manifest_path.with_name('magi_raw.json'))[index]
                paddle=read_payload(manifest_path.with_name('paddle_ocr.json'))[index]
                if len(raw['texts'])!=len(paddle):continue
                if not all(v['box']==b for v,b in zip(paddle,raw['texts'])):continue
                return [dict(box=b,magi_text=t,paddle_text=v['text'],paddle_lines=v['lines'],
                    selected_text=None,paddle_status=v['status'],paddle_error=v.get('error'))
                    for b,t,v in zip(raw['texts'],raw['ocr'],paddle)],str(manifest_path.parent)
            except (OSError,KeyError,ValueError,AssertionError,IndexError,TypeError):
                continue
    return None,None

pages={}
for path in PAGE_FILES:
    records,source=cached_page(path)
    if records is not None:pages[str(path)]=dict(records=records,source=source)
    print('CACHE' if records is not None else 'MISSING',path.name,source or '')


CACHE 01.png C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence\ocr_selection.json
CACHE 02.png C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence\ocr_selection.json
CACHE 03.png C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence\ocr_selection.json


## 1a. Generate missing Magi/Paddle OCR
Only missing pages load local Magi/Paddle models. Models are released before Gemma loads. All detected boxes remain available; agreement < 0.4 is diagnostic, not an early deletion.

In [58]:
def model_identity(path):
    return {str(p.relative_to(path)):[p.stat().st_size,p.stat().st_mtime_ns]
            for p in sorted(Path(path).rglob('*')) if p.is_file()
            and p.suffix in {'.json','.py','.bin','.safetensors','.pdparams','.pdiparams','.yml'}}

missing_pages=[p for p in PAGE_FILES if str(p) not in pages]
identity=None
if missing_pages:
    identity=dict(magi=str(MAGI_MODEL),magi_files=model_identity(MAGI_MODEL),
        paddle_det=str(PADDLE_DET_PATH),det_files=model_identity(PADDLE_DET_PATH),
        paddle_rec=str(PADDLE_REC_PATH),rec_files=model_identity(PADDLE_REC_PATH),algorithm=1)
    for path in list(missing_pages):
        cache_path=CACHE/(digest(dict(page=file_hash(path),models=identity))+'.json')
        if cache_path.is_file():
            try:
                pages[str(path)]=dict(records=read_payload(cache_path),source=str(cache_path))
                missing_pages.remove(path)
            except (OSError,ValueError,KeyError,AssertionError):pass

if missing_pages:
    from transformers import AutoConfig,AutoModel
    assert MAGI_MODEL.is_dir(),f'Missing Magi: {MAGI_MODEL}'
    config=AutoConfig.from_pretrained(str(MAGI_MODEL),trust_remote_code=True,local_files_only=True)
    config.detection_model_config.use_pretrained_backbone=False
    config.disable_ocr=False
    config.disable_detections=False
    model=AutoModel.from_pretrained(str(MAGI_MODEL),config=config,
        trust_remote_code=True,local_files_only=True).to(DEVICE).eval()
    raw_pages=[]
    try:
        for path in missing_pages:
            with Image.open(path) as im:array=np.asarray(im.convert('L').convert('RGB'))
            with torch.inference_mode():
                raw_pages.append(model.do_chapter_wide_prediction([array],{'images':[],'names':[]},do_ocr=True)[0])
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    import paddle
    from paddleocr import PaddleOCR
    assert PADDLE_DET_PATH.is_dir() and PADDLE_REC_PATH.is_dir(),'Set local Paddle model directories.'
    engine=PaddleOCR(text_detection_model_name='PP-OCRv5_server_det',
        text_recognition_model_name='en_PP-OCRv5_mobile_rec',
        text_detection_model_dir=str(PADDLE_DET_PATH),text_recognition_model_dir=str(PADDLE_REC_PATH),
        device='gpu:0' if DEVICE=='cuda' else 'cpu',
        use_doc_orientation_classify=False,use_doc_unwarping=False,use_textline_orientation=False)
    try:
        for path,raw in zip(missing_pages,raw_pages):
            with Image.open(path) as im:im=im.convert('L').convert('RGB')
            records=[]
            for box,magi_text in zip(raw['texts'],raw['ocr']):
                a,b,c,d=box
                bounds=[max(0,math.floor(a)),max(0,math.floor(b)),min(im.width,math.ceil(c)),min(im.height,math.ceil(d))]
                lines=[];error=None
                try:
                    assert bounds[2]>bounds[0] and bounds[3]>bounds[1],'Invalid crop'
                    for prediction in engine.predict(input=np.asarray(im.crop(bounds))):
                        lines.extend(dict(text=str(t),confidence=float(s))
                            for t,s in zip(prediction['rec_texts'],prediction['rec_scores']))
                except Exception as exc:error=str(exc)
                records.append(dict(box=box,magi_text=magi_text,paddle_text=' '.join(v['text'] for v in lines),
                    paddle_lines=lines,selected_text=None,paddle_status='error' if error else 'text' if lines else 'empty',paddle_error=error))
            cache_path=CACHE/(digest(dict(page=file_hash(path),models=identity))+'.json')
            cache_path.write_text(json.dumps(dict(data=records,data_hash=digest(records)),indent=2,ensure_ascii=True),encoding='utf-8')
            pages[str(path)]=dict(records=records,source=str(cache_path))
            print('CREATED',path.name,cache_path)
    finally:
        del engine
        gc.collect()
        if DEVICE=='cuda':paddle.device.cuda.empty_cache()
print('Ready:',len(pages),'pages. Existing pipeline caches were not modified.')


Ready: 3 pages. Existing pipeline caches were not modified.


## 1b. Define Gemma artifact classification
Only Paddle text is sent to Gemma, using the same prompt/examples as the artifact test. Punctuation-only Paddle is Text directly. Empty Paddle uses punctuation-only Magi; other empty-Paddle boxes become Artifact. Unexpected model answers become Review. Edit the prompt/examples in the next cell.

In [59]:
SYSTEM_PROMPT=(
    'Classify the Paddle OCR text of one manga text box. Return exactly Text or Artifact. '
    'Text includes dialogue, narration, names, Japanese words/names, unfinished sentences, '
    'multiple sentences and multiple lines. Profanity and figurative language are allowed. '
    'Integers, integer-only text and numbers within meaningful dialogue are Text. '
    'Punctuation-only expressions are Text. Vocal reactions, screams, disgust, hesitation '
    'and stretched spoken words such as AAAAHH!!!, NOO!! and EWWW! are Text. '
    'Distinguish a human vocal reaction from a written environmental or action sound effect: '
    'Whoosh, BAM, CLANG and similar sound effects are Artifact. '
    'Artifact also includes nonsensical OCR noise, scan-group watermarks, website links '
    'and isolated code-like strings such as 2-B, 3-2 and V45S. '
    'Single Text Char are Artifacts '
    'Do not reject an unfamiliar name or a meaningful sentence simply because it contains '
    'a number or identifier. If uncertain, prefer Text. Do not correct, rewrite or explain the text.')

# One Paddle OCR input and one label per example.
EXAMPLES = [
    # Human vocal reactions and stretched dialogue are legitimate text.
    ('GAAAAAAHH!!!', 'Text'),
    ('AAAAHH!!!', 'Text'),
    ('AAAH!', 'Text'),
    ('NOO!!', 'Text'),
    ('EWWW!', 'Text'),
    ('UGH... THAT SMELLS AWFUL!', 'Text'),
    ('HUH?!', 'Text'),
    ('OH...', 'Text'),
    ('UM... I THINK SO.', 'Text'),
    ('HAHAHA!', 'Text'),
    ('OW! THAT HURTS!', 'Text'),
    ('YAAAY!!', 'Text'),
    # Dialogue, narration, names, profanity and unfinished sentences.
    ('I SAID TO NOT DO THAT.', 'Text'),
    ('But our characters were unaware', 'Text'),
    ('NANIKAWA', 'Text'),
    ('SHIROGANE, WAIT!', 'Text'),
    ("IT'S A GODDAMN MIRACLE, THE MONEY JUST VANISHED INTO THIN AIR.", 'Text'),
    ('IF SHE SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,', 'Text'),
    # Multiple sentences and OCR line breaks do not make a box an artifact.
    ('WAIT! WHERE DID EVERYONE GO? I WAS JUST HERE.', 'Text'),
    ('I TRUSTED YOU. NOW TELL ME WHAT HAPPENED.', 'Text'),
    ('WE HAVE TO LEAVE.\nTHE TRAIN IS ALREADY HERE!', 'Text'),
    ('Among the six who possessed the aptitude,\nSaiko-chan had the highest aptitude.', 'Text'),
    # Plain integers and numbers in meaningful lines are allowed.
    ('123', 'Text'),
    ('42', 'Text'),
    ('0', 'Text'),
    ('I HAVE 2 QUESTIONS.', 'Text'),
    ('WE HAVE 3 MINUTES LEFT!', 'Text'),
    ('HE PAID 500 YEN FOR IT.', 'Text'),
    ('I AM IN CLASS 2-B.', 'Text'),
    ('THE SCORE IS 3-2. WE CAN STILL WIN!', 'Text'),
    # Punctuation-only expressions are also protected by a Python rule.
    ('...', 'Text'),
    ('!!!!', 'Text'),
    ('...!!!', 'Text'),
    ('?!', 'Text'),
    
    # Written action/environment sounds, rather than human utterances.
    ('もしもし', 'Artifact'),
    ('しし', 'Artifact'),
    ('も', 'Artifact'),
    ('a ~', 'Artifact'),
    ('~~~~~ ', 'Artifact'),
    ('~~~~~ ', 'Artifact'),
    ('Whoosh!!', 'Artifact'),
    ('VOOSH', 'Artifact'),
    ('KIRK', 'Artifact'),
    ('bEE', 'Artifact'),
    ('WFAG', 'Artifact'),
    ('VOOOSH', 'Artifact'),
    ('O', 'Artifact'),
    ('K', 'Artifact'),
    ('AKK', 'Artifact'),
    ('HUPP', 'Artifact'),
    ('CRASHHH', 'Artifact'),
    ('SHHH', 'Artifact'),
    ('KRASHHH', 'Artifact'),
    ('BAM', 'Artifact'),
    ('CLANG!', 'Artifact'),
    ('THUD', 'Artifact'),
    ('CRASH!!', 'Artifact'),
    ('RUSTLE RUSTLE', 'Artifact'),
    ('CLICK', 'Artifact'),
    ('SPLASH!', 'Artifact'),
    ('SFX', 'Artifact'),
    ('SKX', 'Artifact'),
    # Watermarks, links and corrupted OCR.
    ('www.google.com', 'Artifact'),
    ('https://mangastream.com/read/123', 'Artifact'),
    ('MANGA STREAM DO NOT MIRROR', 'Artifact'),
    ('M4NGA STRE4M', 'Artifact'),
    ('DO NOT REDISTRIBUTE', 'Artifact'),
    ('oo', 'Artifact'),
    ('Chapter 32', 'Artifact'),
    ('Chapter 12', 'Artifact'),
    ('Chapter Over', 'Artifact'),
    ('Chapter End', 'Artifact'),
    ('To be Continued', 'Artifact'),
    ('Chapter', 'Artifact'),
    ('XZQWZXQ', 'Artifact'),
    # Standalone code-like strings requested as artifacts for this test.
    ('2-B', 'Artifact'),
    ('3-2', 'Artifact'),
    ('V45S', 'Artifact'),
]


def punctuation_only(text):
    text=str(text or '').strip()
    return bool(text) and all(c.isspace() or unicodedata.category(c).startswith('P') for c in text)


def direct_decision(paddle,magi=''):
    # Only punctuation may be recovered from Magi when Paddle finds no text.
    if not str(paddle or '').strip():
        if punctuation_only(magi):return 'Text','Magi punctuation fallback: Paddle OCR empty'
        return 'Artifact','Paddle OCR empty; no Magi punctuation fallback'
    if punctuation_only(paddle):return 'Text','punctuation-only override'
    return None,None


def paddle_prompt(paddle):
    return json.dumps({'Paddle OCR':str(paddle or '')},ensure_ascii=False)


def classify_paddle(paddle,model,tokenizer):
    messages=[dict(role='system',content=SYSTEM_PROMPT)]
    for example,label in EXAMPLES:
        messages.extend([dict(role='user',content=paddle_prompt(example)),dict(role='assistant',content=label)])
    messages.append(dict(role='user',content=paddle_prompt(paddle)))
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,
        tokenize=True,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(**inputs,do_sample=False,max_new_tokens=8)
    answer=tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    # Unexpected model output stays visible as Review instead of causing deletion.
    label={'text':'Text','artifact':'Artifact'}.get(answer.casefold(),'Review')
    return dict(label=label,raw_answer=answer,reason='Gemma classification' if label!='Review' else 'unexpected Gemma answer')


## 1c. Run local Gemma here
Load `C:/MODELS/gemma-3-4b-it` once in 4-bit NF4 when decisions are missing, and release it before YOLO/tail models. Matching classification caches belong to this combined notebook and include prompt/examples/model identity. Set `FORCE_GEMMA=True` to regenerate. No `ArtifactGemmaTest/comparison.json` is read.

In [60]:
MODEL_ID=model_identity(GEMMA_MODEL)
CLASSIFIER_SETTINGS=dict(model=str(GEMMA_MODEL),files=MODEL_ID,system=SYSTEM_PROMPT,
    examples=EXAMPLES,algorithm=3,input_source='paddle_only',empty_paddle='magi_punctuation_else_delete',max_new_tokens=8,quantization='nf4',
    transformers=__import__('transformers').__version__)
DECISION_CACHE=CACHE/'decisions'
DECISION_CACHE.mkdir(parents=True,exist_ok=True)
rows=[]
for path in PAGE_FILES:
    for index,record in enumerate(pages[str(path)]['records']):
        a=str(record.get('magi_text') or '');b=str(record.get('paddle_text') or '')
        na=''.join(re.findall('[a-z0-9]',a.casefold()))
        nb=''.join(re.findall('[a-z0-9]',b.casefold()))
        agreement=SequenceMatcher(None,na,nb).ratio() if na and nb else None
        rows.append(dict(page_file=str(path),box_id=f'PG{int(path.stem)}:T{index+1}',
            text_index=index,box=record['box'],magi_text=a,paddle_text=b,
            effective_text=b if b.strip() else a if punctuation_only(a) else '',
            text_source='paddle' if b.strip() else 'magi_punctuation' if punctuation_only(a) else 'none',
            paddle_lines=record.get('paddle_lines',[]),paddle_error=record.get('paddle_error'),
            agreement=agreement,low_agreement=agreement is not None and agreement<0.4,
            key=digest(dict(settings=CLASSIFIER_SETTINGS,paddle=b,magi_fallback=a if not b.strip() else None))))

pending={}
decisions={}
for row in rows:
    label,reason=direct_decision(row['paddle_text'],row['magi_text'])
    if label:
        decisions[row['key']]=dict(label=label,reason=reason,raw_answer='')
        continue
    cache_path=DECISION_CACHE/(row['key']+'.json')
    if cache_path.is_file() and not FORCE_GEMMA:
        try:
            saved=read_payload(cache_path)
            assert saved['label'] in {'Text','Artifact','Review'}
            decisions[row['key']]=saved
            continue
        except (OSError,ValueError,KeyError,AssertionError):pass
    pending[row['key']]=row

print('Boxes:',len(rows),'uncached unique Paddle texts:',len(pending))
if pending:
    assert GEMMA_MODEL.is_dir() and MODEL_ID,'Missing local Gemma model'
    assert torch.cuda.is_available(),'4-bit Gemma requires a CUDA kernel in this notebook'
    from transformers import AutoTokenizer,Gemma3ForConditionalGeneration,BitsAndBytesConfig
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(str(GEMMA_MODEL),local_files_only=True)
    model=Gemma3ForConditionalGeneration.from_pretrained(str(GEMMA_MODEL),local_files_only=True,
        device_map='auto',dtype=dtype,quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=dtype)).eval()
    try:
        for number,(key,row) in enumerate(pending.items(),1):
            decision=classify_paddle(row['paddle_text'],model,tokenizer)
            decisions[key]=decision
            (DECISION_CACHE/(key+'.json')).write_text(json.dumps(
                dict(data=decision,data_hash=digest(decision)),ensure_ascii=False,indent=2),encoding='utf-8')
            print(number,'/',len(pending),row['box_id'],decision['label'])
    finally:
        del model,tokenizer
        gc.collect();torch.cuda.empty_cache()

for row in rows:
    row.update(decisions[row['key']])
    row['delete']=row['label']=='Artifact'
comparison=pd.DataFrame(rows,columns=['box_id','page_file','text_index','box','magi_text','paddle_text',
    'effective_text','text_source','paddle_lines','paddle_error','agreement','low_agreement','key','label','reason','raw_answer','delete'])
display(comparison[['box_id','magi_text','paddle_text','agreement','label','reason','raw_answer']])
print(comparison['label'].value_counts().to_dict())

# Stage 1 uses these fresh/cached-in-this-notebook decisions directly.
for row in rows:
    row['contour_eligible']=row['label']!='Artifact' and bool(row['effective_text'].strip())
ARTIFACT_RESULTS=OUTPUT/'gemma_results.json'
ARTIFACT_RESULTS.write_text(json.dumps(rows,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')


Boxes: 41 uncached unique Paddle texts: 39


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

1 / 39 PG1:T1 Text
2 / 39 PG1:T2 Text
3 / 39 PG1:T3 Text
4 / 39 PG1:T4 Text
5 / 39 PG1:T5 Artifact
6 / 39 PG1:T6 Text
7 / 39 PG1:T7 Text
8 / 39 PG1:T8 Text
9 / 39 PG1:T9 Text
10 / 39 PG1:T10 Text
11 / 39 PG1:T11 Text
12 / 39 PG1:T12 Artifact
13 / 39 PG1:T13 Text
14 / 39 PG1:T14 Text
15 / 39 PG2:T1 Text
16 / 39 PG2:T2 Text
17 / 39 PG2:T3 Text
18 / 39 PG2:T4 Text
19 / 39 PG2:T5 Text
20 / 39 PG2:T6 Text
21 / 39 PG2:T7 Text
22 / 39 PG2:T8 Text
23 / 39 PG2:T9 Text
24 / 39 PG2:T10 Text
25 / 39 PG2:T11 Text
26 / 39 PG2:T12 Text
27 / 39 PG2:T13 Text
28 / 39 PG3:T1 Text
29 / 39 PG3:T2 Text
30 / 39 PG3:T3 Text
31 / 39 PG3:T4 Text
32 / 39 PG3:T5 Text
33 / 39 PG3:T6 Text
34 / 39 PG3:T7 Text
35 / 39 PG3:T8 Text
36 / 39 PG3:T9 Text
37 / 39 PG3:T10 Text
38 / 39 PG3:T11 Text
39 / 39 PG3:T12 Text


,box_id,magi_text,paddle_text,agreement,label,reason,raw_answer
0,PG1:T1,...this is bad.,...THIS IS BAD.,1.000000,Text,Gemma classification,Text
1,PG1:T2,It looks like that person is going to follow D...,IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S...,0.987654,Text,Gemma classification,Text
2,PG1:T3,If she foods in luring out one person in this ...,IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN ...,0.906977,Text,Gemma classification,Text
3,PG1:T4,Then not will probably leave.,THEN NUT WILL PROBABLY LEAVE.,0.958333,Text,Gemma classification,Text
4,PG1:T5,...,444,NaN,Artifact,Gemma classification,Artifact
5,PG1:T6,Sassan.,SASSAN.,1.000000,Text,Gemma classification,Text
6,PG1:T7,Hmm?,HM?,0.800000,Text,Gemma classification,Text
7,PG1:T8,"About Saiko,","ABOUT SAIKO,",1.000000,Text,Gemma classification,Text
8,PG1:T9,I get the feeling,I GET THE FEELING,1.000000,Text,Gemma classification,Text
9,PG1:T10,that you are being soft on her...,THAT YOU ARE BEING SOFT ON HER...,1.000000,Text,Gemma classification,Text


{'Text': 38, 'Artifact': 3}


44650

## 2. Before contour: Gemma decisions
One page at a time. Green Text, red Artifact, orange Review. No automatic crop duplication. The after-contour viewer below displays the independent role predictions.

In [61]:
# Exactly one page per viewer. Render a PNG explicitly to avoid backend double display.
from io import BytesIO
from IPython.display import Image as DisplayImage, clear_output
COLORS={'Text':'limegreen','Artifact':'red','Review':'darkorange'}


def render_result_page(records,page_index,title):
    path=PAGE_FILES[page_index]
    with Image.open(path) as image:
        fig,ax=plt.subplots(figsize=(10,14))
        ax.imshow(image.convert('RGB'))
    page_rows=[r for r in records if r['page_file']==str(path)]
    for row in page_rows:
        x1,y1,x2,y2=row['box'];color=COLORS[row['label']]
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,linewidth=2))
        ax.text(x1,y1,row['box_id']+' '+row['label'],fontsize=8,
            bbox=dict(facecolor=color,alpha=.8),color='black')
    ax.set_title(title+' | '+path.name+' | green Text / red Artifact / orange Review')
    ax.axis('off')
    buffer=BytesIO()
    try:
        fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:
        plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


def make_page_viewer(records,title,key):
    import ipywidgets as widgets
    # Close the previous instance when this cell is rerun.
    registry=globals().setdefault('_combined_viewers',{})
    old=registry.pop(key,None)
    if old:
        old['picker'].unobserve_all()
        for widget in old.values():widget.close()
    picker=widgets.Dropdown(options=[(p.name,i) for i,p in enumerate(PAGE_FILES)],
        value=0,description='Page')
    output=widgets.Output()

    def refresh(change=None):
        with output:
            clear_output(wait=True)
            render_result_page(records,picker.value,title)

    picker.observe(refresh,names='value')
    panel=widgets.VBox([widgets.HTML(value='<b>'+title+'</b>'),picker,output])
    registry[key]=dict(picker=picker,output=output,panel=panel)
    display(panel)
    refresh()



if PAGE_FILES:make_page_viewer(rows,'Before contour: Gemma labels','before')


## 3. Reuse or create YOLO bubble candidates
Local bubble weights only. Discovery checks full-pipeline candidate caches by page hash, dimensions, weight hash and inference settings. Newly generated candidates are saved separately. This runs on all selected pages, including those containing Gemma Artifacts.

In [62]:
def matching_bubble_cache(path,weights_hash):
    sha=file_hash(path)
    with Image.open(path) as image:size=image.size
    # Reuse full-pipeline candidates before resorting to new detection.
    root=PROJECT/'paddleOCR-Magi/cache_contour_full_inference'
    for manifest_path in root.rglob('manifest.json'):
        try:
            manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
            settings=manifest['stages']['contour_bubbles']['settings']
            if settings['weights_sha256']!=weights_hash or settings['settings']!=BUBBLE_SETTINGS:continue
            matches=[i for i,page in enumerate(manifest['inputs']['pages'])
                if page['sha256']==sha and (page['width'],page['height'])==size]
            if len(matches)==1:
                return read_payload(manifest_path.with_name('contour_bubbles.json'))[matches[0]]['candidates']
        except (OSError,KeyError,ValueError,AssertionError,IndexError,TypeError):continue
    return None

assert BUBBLE_MODEL.is_file(),f'Missing local YOLO: {BUBBLE_MODEL}'
weights_hash=file_hash(BUBBLE_MODEL)
bubble_cache=CACHE/'bubbles'
bubble_cache.mkdir(parents=True,exist_ok=True)
bubble_pages={};detector=None
try:
    for path in PAGE_FILES:
        key=digest(dict(page=file_hash(path),weights=weights_hash,settings=BUBBLE_SETTINGS))
        cache_path=bubble_cache/(key+'.json')
        candidates=None;source=None
        if cache_path.is_file() and not FORCE_BUBBLES:
            try:candidates=read_payload(cache_path);source=str(cache_path)
            except (OSError,ValueError,KeyError,AssertionError):pass
        # The artifact notebook's previous bubble pass uses the same identity key.
        artifact_cache=PIPELINE_ROOT/'cache_artifact_gemma_test/punctuation_bubbles'/(key+'.json')
        if candidates is None and artifact_cache.is_file() and not FORCE_BUBBLES:
            try:candidates=read_payload(artifact_cache);source=str(artifact_cache)
            except (OSError,ValueError,KeyError,AssertionError):pass
        if candidates is None and not FORCE_BUBBLES:
            candidates=matching_bubble_cache(path,weights_hash)
            if candidates is not None:source='compatible full-pipeline cache'
        if candidates is None:
            if detector is None:
                from ultralytics import YOLO
                detector=YOLO(str(BUBBLE_MODEL))
            with Image.open(path) as image:
                image=image.convert('RGB');width,height=image.size
                prediction=detector.predict(source=image,imgsz=BUBBLE_SETTINGS['imgsz'],
                    conf=BUBBLE_SETTINGS['confidence'],device=0 if DEVICE=='cuda' else 'cpu',verbose=False)[0]
            candidates=[]
            for box,score in zip(prediction.boxes.xyxy.cpu().tolist(),prediction.boxes.conf.cpu().tolist()):
                a,b,c,d=box
                bounds=[max(0,math.floor(a)),max(0,math.floor(b)),min(width,math.ceil(c)),min(height,math.ceil(d))]
                if bounds[2]>bounds[0] and bounds[3]>bounds[1]:candidates.append(dict(box=bounds,confidence=float(score)))
            candidates.sort(key=lambda b:(b['box'][1],-b['box'][0]))
            del prediction
            source='local YOLO'
        cache_path.write_text(json.dumps(dict(data=candidates,data_hash=digest(candidates)),indent=2),encoding='utf-8')
        bubble_pages[str(path)]=candidates
        print(path.name,len(candidates),'candidates |',source)
finally:
    if detector is not None:
        del detector;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


01.png 15 candidates | compatible full-pipeline cache
02.png 16 candidates | compatible full-pipeline cache
03.png 10 candidates | compatible full-pipeline cache


## Stage 1: artifact filtering and punctuation fallback verification
Keep Gemma Artifact decisions deleted. Only empty Paddle + punctuation-only Magi fallbacks undergo the bubble-mask check (20% padded crop, normal/faint checks, visible image-edge support, no suspected leak). No qualifying mask means delete. No Gemma Artifact rescue. Stage 2 receives only the remaining usable texts.

In [63]:
# Stage 1: retain the agreed punctuation-only contour gate; no Artifact rescue.
assert CONTOUR_HELPER.is_file(),f'Missing contour helper: {CONTOUR_HELPER}'
spec=importlib.util.spec_from_file_location('artifact_combined_contour',CONTOUR_HELPER)
contour_model=importlib.util.module_from_spec(spec)
spec.loader.exec_module(contour_model)
stage1_audit=[]
for row in rows:
    row['gemma_label']=row['label']
    row['stage1_bubble_checked']=False
    row['stage1_bubble_found']=None
for path in PAGE_FILES:
    page_rows=[r for r in rows if r['page_file']==str(path)]
    text_boxes=[r['box'] for r in page_rows]
    with Image.open(path) as original:image=original.convert('RGB')
    for index,row in enumerate(page_rows):
        if (row['label']=='Artifact' or str(row.get('paddle_text') or '').strip()
            or not punctuation_only(row.get('magi_text'))):continue
        found=False
        for candidate in bubble_pages[str(path)]:
            a,b,c,d=row['box'];x,y,u,v=candidate['box']
            overlap=max(0,min(c,u)-max(a,x))*max(0,min(d,v)-max(b,y))
            if overlap/max(1,(c-a)*(d-b))<0.8:continue
            checked=contour_model.find_background_regions(candidate,image,text_boxes,index)
            region=checked['best']
            if (region is not None and not region['leak_suspected']
                and region['visible_points']>=contour_model.REGION['min_visible_points']):
                found=True;break
        row['stage1_bubble_checked']=True
        row['stage1_bubble_found']=found
        if not found:
            row.update(label='Artifact',delete=True,contour_eligible=False,
                reason='Stage 1: empty Paddle + Magi punctuation has no qualifying bubble mask')
        stage1_audit.append(dict(box_id=row['box_id'],gemma_label=row['gemma_label'],
            bubble_mask_found=found,artifact_stage_label=row['label'],reason=row['reason']))
for row in rows:
    row['artifact_stage_label']=row['label']
    row['contour_eligible']=row['label']!='Artifact' and bool(row['effective_text'].strip())
display(pd.DataFrame(stage1_audit))
print('Stage 1 punctuation fallbacks checked:',len(stage1_audit),
    '| Removed:',sum(not r['bubble_mask_found'] for r in stage1_audit))


""


Stage 1 punctuation fallbacks checked: 0 | Removed: 0


## 3a. Magi tail evidence
Reuse raw Magi page predictions with explicit text-to-tail links. Match page hashes and original box indices/coordinates; fallback predictions require unique IoU >= 0.95 alignment. A tail somewhere on the page does not affect unrelated texts. No tail affinity threshold is used. Deleted Artifacts never re-enter classification.

In [64]:
# Explicit Magi text-to-tail links force SPOKEN; a tail elsewhere on the page does not.
MAGI_MODEL=Path('C:/MODELS/magiv2')
TAIL_FORCE_SPOKEN=True
TAIL_MIN_BOX_IOU=0.95
TAIL_CACHE=CACHE/'magi_tails'
TAIL_CACHE.mkdir(parents=True,exist_ok=True)
TAIL_CACHE_ROOTS=[PIPELINE_ROOT/'cache_contour_full_inference',
    PROJECT/'FINETUNING-EXPERMENTS/NarrationModelwBubbles/cache_selected_ocr/full_pipeline',
    PIPELINE_ROOT/'cache_inference']


def tail_cache_compatible(raw,page_rows):
    if 'tails' not in raw or 'text_tail_associations' not in raw:return False
    boxes=raw.get('texts',[])
    for row in page_rows:
        index=row['text_index']
        if index>=len(boxes) or not np.allclose(boxes[index],row['box'],atol=0.05,rtol=0):return False
    return True


def load_tail_page(path,page_rows):
    sha=file_hash(path)
    with Image.open(path) as image:size=image.size
    for root in TAIL_CACHE_ROOTS:
        for manifest_path in root.rglob('manifest.json'):
            try:
                manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
                matches=[i for i,page in enumerate(manifest['inputs']['pages'])
                    if page['sha256']==sha and (page['width'],page['height'])==size]
                if len(matches)!=1:continue
                raw=read_payload(manifest_path.with_name('magi_raw.json'))[matches[0]]
                if tail_cache_compatible(raw,page_rows):return raw,str(manifest_path.parent)
            except (OSError,KeyError,ValueError,AssertionError,IndexError,TypeError):continue
    return None,None


def box_iou(a,b):
    intersection=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    return intersection/max(1,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-intersection)


magi_tail_pages={};tail_links={};tail_sources={};magi_model=None
try:
    for path in PAGE_FILES:
        page_rows=[r for r in rows if r['page_file']==str(path) and r['contour_eligible']]
        if not page_rows:continue
        raw,source=load_tail_page(path,page_rows)
        # If the source OCR cache lacks tails, use a separate local prediction cache.
        local_key=digest(dict(page=file_hash(path),model=str(MAGI_MODEL),
            config=file_hash(MAGI_MODEL/'config.json'),algorithm=1))
        local_path=TAIL_CACHE/(local_key+'.json')
        if raw is None and local_path.is_file():
            raw=read_payload(local_path);source=str(local_path)
        if raw is None:
            if magi_model is None:
                from transformers import AutoConfig,AutoModel
                config=AutoConfig.from_pretrained(str(MAGI_MODEL),trust_remote_code=True,local_files_only=True)
                config.detection_model_config.use_pretrained_backbone=False
                config.disable_ocr=False;config.disable_detections=False
                magi_model=AutoModel.from_pretrained(str(MAGI_MODEL),config=config,
                    trust_remote_code=True,local_files_only=True).to(DEVICE).eval()
            with Image.open(path) as image:array=np.asarray(image.convert('L').convert('RGB'))
            with torch.inference_mode():
                prediction=magi_model.do_chapter_wide_prediction([array],{'images':[],'names':[]},do_ocr=False)[0]
            def plain(value):
                if isinstance(value,torch.Tensor):return value.detach().cpu().tolist()
                if isinstance(value,np.ndarray):return value.tolist()
                if isinstance(value,dict):return {k:plain(v) for k,v in value.items()}
                if isinstance(value,(list,tuple)):return [plain(v) for v in value]
                return value
            raw=plain({k:prediction[k] for k in ['texts','tails','text_tail_associations']})
            local_path.write_text(json.dumps(dict(data=raw,data_hash=digest(raw)),indent=2),encoding='utf-8')
            source='local Magi prediction'
        assert 'tails' in raw and 'text_tail_associations' in raw,'Magi prediction lacks tail associations'
        magi_tail_pages[str(path)]=raw;tail_sources[str(path)]=source
        used=set()
        for row in page_rows:
            index=row['text_index']
            if not (index<len(raw['texts']) and np.allclose(raw['texts'][index],row['box'],atol=0.05,rtol=0)):
                scores=[box_iou(row['box'],box) for box in raw['texts']]
                assert scores and max(scores)>=TAIL_MIN_BOX_IOU,f"Cannot align Magi tails safely: {row['box_id']}"
                index=int(np.argmax(scores))
                assert sum(score>=TAIL_MIN_BOX_IOU for score in scores)==1,f"Ambiguous tail alignment: {row['box_id']}"
            assert index not in used,f"Duplicate tail alignment: {row['box_id']}"
            used.add(index)
            links=[int(tail) for text,tail in raw['text_tail_associations'] if int(text)==index]
            assert all(0<=tail<len(raw['tails']) for tail in links),'Invalid Magi tail index'
            tail_links[row['box_id']]=links
        print(path.name,'tail-linked retained texts:',sum(bool(tail_links[r['box_id']]) for r in page_rows),'|',source)
finally:
    if magi_model is not None:
        del magi_model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


01.png tail-linked retained texts: 4 | C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence
02.png tail-linked retained texts: 2 | C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence
03.png tail-linked retained texts: 1 | C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_contour_full_inference\test\seq_14469aeb38963085\sequence


## Stage 2: grouping and SPOKEN/NARRATED classification
Use the existing individual outline checks, faint retries, overlapping merge candidates and concatenation in original reading order. Then surrounding white/solid regions, visible curves, sharp sides and rectangular shapes determine SPOKEN/NARRATED. Gemma Artifacts do not participate in candidate selection, text matching or merged text. No usable OCR means no contour role.

In [65]:
assert CONTOUR_HELPER.is_file(),f'Missing contour helper: {CONTOUR_HELPER}'
spec=importlib.util.spec_from_file_location('artifact_combined_contour',CONTOUR_HELPER)
contour_model=importlib.util.module_from_spec(spec)
spec.loader.exec_module(contour_model)
analyses={};debug_pages={};groups=[]
for path in PAGE_FILES:
    page_rows=sorted([r for r in rows if r['page_file']==str(path) and r['contour_eligible']],key=lambda r:r['text_index'])
    texts=[dict(box_id=r['box_id'],text_index=r['text_index'],box=r['box'],
        text=r['effective_text'],magi_text=r.get('magi_text',''),paddle_text=r.get('paddle_text',''),
        ocr_agreement=r.get('agreement')) for r in page_rows]
    with Image.open(path) as image:
        analysis,debug=contour_model.analyze_page(bubble_pages[str(path)],image.convert('RGB'),texts,int(path.stem))
    analyses[str(path)]=analysis;debug_pages[str(path)]=debug
    for group in analysis['rows']:
        group['page_file']=str(path)
        groups.append(group)
    print(path.name,len(texts),'text boxes ->',len(analysis['rows']),'groups')

# A linked tail on any surviving member makes the entire concatenated group SPOKEN.
for group in groups:
    group['contour_text_role']=group['text_role']
    group['contour_original_reason']=group['reason']
    group['tail_member_box_ids']=[bid for bid in group['box_ids'] if tail_links.get(bid)]
    group['tail_ids']=sorted({tail for bid in group['box_ids'] for tail in tail_links.get(bid,[])})
    group['tail_override']=bool(TAIL_FORCE_SPOKEN and group['tail_ids'])
    if group['tail_override']:
        group['text_role']='SPOKEN'
        group['reason']='Magi text-to-tail association forces SPOKEN'


01.png 13 text boxes -> 11 groups
02.png 13 text boxes -> 10 groups
03.png 12 text boxes -> 11 groups


## 5. Inspect remaining text roles and deleted artifacts
Original Gemma labels remain available in the per-box audit. Deleted Artifacts have `NOT_TESTED` as their contour role. Remaining texts receive SPOKEN/NARRATED; no-region cases default to NARRATED and are marked as lacking visual support. These are predicted roles, not ground truth.

In [66]:
lookup={r['box_id']:r for r in rows}
by_member={}
individual_outlines={}
for analysis in analyses.values():
    for candidate in analysis['grouping']['individual']:
        if candidate['boundary_found']:
            for bid in candidate['text_ids']:individual_outlines.setdefault(bid,[]).append(candidate['candidate_id'])
for group in groups:
    members=[lookup[bid] for bid in group['box_ids']]
    group['member_gemma_labels']=[r['label'] for r in members]
    group['artifact_box_ids']=[r['box_id'] for r in members if r['label']=='Artifact']
    group['has_artifact_member']=bool(group['artifact_box_ids'])
    group['region_found']=group.get('crop_bounds') is not None
    group['visual_support']=bool(group['region_found'] and not group.get('leak_suspected',False)
        and group.get('visible_points',0)>=contour_model.REGION['min_visible_points'])
    for bid in group['box_ids']:
        assert bid not in by_member,'Text assigned to more than one group'
        by_member[bid]=group
comparison_rows=[]
for original in rows:
    row=dict(original);group=by_member.get(row['box_id'])
    role=group['text_role'] if group else 'NOT_TESTED'
    row.update(tail_ids=tail_links.get(row['box_id'],[]),
        has_magi_tail=bool(tail_links.get(row['box_id'])),
        tail_override=bool(group and group['tail_override']),
        contour_text_role=group['contour_text_role'] if group else 'NOT_TESTED',
        gemma_label=row['gemma_label'],group_id=group['box_id'] if group else None,
        group_box_ids=group['box_ids'] if group else [],text_role=role,
        contour_reason=group['reason'] if group else ('Removed: Gemma Artifact' if row['label']=='Artifact' else 'No usable Paddle text or Magi punctuation fallback'),
        grouping_source=group['grouping_source'] if group else None,
        individual_outline=bool(individual_outlines.get(row['box_id'])),
        region_found=bool(group and group['region_found']),
        visual_support=bool(group and group['visual_support']),
        contour_needs_review=bool(group and group['needs_review']),
        artifact_with_contour_role=row['label']=='Artifact' and role in {'SPOKEN','NARRATED'},
        artifact_with_visual_support=row['label']=='Artifact' and bool(group and group['visual_support']))
    comparison_rows.append(row)
assert set(by_member)=={r['box_id'] for r in rows if r['contour_eligible']}
combined=pd.DataFrame(comparison_rows)
display(pd.DataFrame([dict(page=p.name,boxes=sum(r['page_file']==str(p) for r in rows),
    groups=sum(g['page_file']==str(p) for g in groups),
    artifact_role=sum(r['page_file']==str(p) and r['artifact_with_contour_role'] for r in comparison_rows),
    artifact_visual_support=sum(r['page_file']==str(p) and r['artifact_with_visual_support'] for r in comparison_rows)) for p in PAGE_FILES]))
if len(combined):
    display(combined[['box_id','effective_text','gemma_label','group_id','text_role','individual_outline','has_magi_tail','tail_override',
        'region_found','visual_support','artifact_with_contour_role','contour_reason']])
    print('Removed after stage 1:',sum(r['artifact_stage_label']=='Artifact' for r in rows))


,page,boxes,groups,artifact_role,artifact_visual_support
0,01.png,15,11,0,0
1,02.png,14,10,0,0
2,03.png,12,11,0,0


,box_id,effective_text,gemma_label,group_id,text_role,individual_outline,has_magi_tail,tail_override,region_found,visual_support,artifact_with_contour_role,contour_reason
0,PG1:T1,...THIS IS BAD.,Text,PG1:T1,SPOKEN,True,False,False,True,True,False,white region with substantial curved sides; ov...
1,PG1:T2,IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S...,Text,PG1:M1,SPOKEN,False,True,True,True,True,False,Magi text-to-tail association forces SPOKEN
2,PG1:T3,IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN ...,Text,PG1:M1,SPOKEN,False,True,True,True,True,False,Magi text-to-tail association forces SPOKEN
3,PG1:T4,THEN NUT WILL PROBABLY LEAVE.,Text,PG1:T4,SPOKEN,True,False,False,True,True,False,white region with substantial curved sides; ov...
4,PG1:T5,444,Artifact,NaN,NOT_TESTED,False,False,False,False,False,False,Removed: Gemma Artifact
5,PG1:T6,SASSAN.,Text,PG1:T6,SPOKEN,True,False,False,True,True,False,white region with substantial curved sides; ov...
6,PG1:T7,HM?,Text,PG1:T7,SPOKEN,True,False,False,True,True,False,white region with meaningful visible curved sides
7,PG1:T8,"ABOUT SAIKO,",Text,PG1:T8,SPOKEN,True,True,True,True,True,False,Magi text-to-tail association forces SPOKEN
8,PG1:T9,I GET THE FEELING,Text,PG1:M2,SPOKEN,False,False,False,True,True,False,white region with substantial curved sides; ov...
9,PG1:T10,THAT YOU ARE BEING SOFT ON HER...,Text,PG1:M2,SPOKEN,False,False,False,True,True,False,white region with substantial curved sides; ov...


Removed after stage 1: 3


## 6. After artifact removal: contour role viewer
Only remaining texts appear: yellow SPOKEN, grey NARRATED. Visible contour segments are drawn; group rectangles indicate concatenated text membership. One page at a time. Optional manual crop/mask inspection: `inspect_group('PG1:M1')`.

`[TAIL]` marks the hard SPOKEN override; linked tails appear in purple. Original contour roles remain in the exports.

In [67]:
def render_combined_page(page_index=0):
    path=PAGE_FILES[page_index]
    with Image.open(path) as image:
        fig,ax=plt.subplots(figsize=(10,14));ax.imshow(image.convert('RGB'))
    for group in [g for g in groups if g['page_file']==str(path)]:
        color='gold' if group['text_role']=='SPOKEN' else 'grey'
        x1,y1,x2,y2=group['text_box']
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,linewidth=2))
        for segment in group['visible_segments']:
            points=np.asarray(segment);ax.plot(points[:,0],points[:,1],color=color,linewidth=1.5)
        ax.text(x1,y1,group['box_id']+' '+group['text_role']+(' [TAIL]' if group['tail_override'] else ''),fontsize=8,bbox=dict(facecolor=color,alpha=.8))
    for row in [r for r in comparison_rows if r['page_file']==str(path) and r['contour_eligible']]:
        x1,y1,x2,y2=row['box']
        if row['gemma_label']=='Artifact':
            ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='red',linewidth=1,linestyle='--'))
            ax.text(x1,y2,row['box_id']+' GEMMA: ARTIFACT',fontsize=7,color='red',bbox=dict(facecolor='white',alpha=.85))
        elif row['text_role']=='NOT_TESTED':
            ax.text(x1,y1,row['box_id']+' NOT TESTED',fontsize=8,color='darkorange')
    # Draw only tails linked to retained groups.
    raw=magi_tail_pages.get(str(path),{})
    tail_ids={tail for group in groups if group['page_file']==str(path) for tail in group['tail_ids']}
    for tail in tail_ids:
        x1,y1,x2,y2=raw['tails'][tail]
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='purple',linewidth=1.5))
    ax.set_title('After artifact removal + contour | '+path.name+' | yellow SPOKEN / grey NARRATED')
    ax.axis('off');buffer=BytesIO()
    try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
    finally:plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))


import ipywidgets as widgets
old=globals().get('_combined_after_widgets')
if old:
    old[0].unobserve_all()
    for widget in old:widget.close()
after_picker=widgets.Dropdown(options=[(p.name,i) for i,p in enumerate(PAGE_FILES)],value=0,description='Page')
after_output=widgets.Output()

def refresh_after(change=None):
    with after_output:
        clear_output(wait=True)
        render_combined_page(after_picker.value)

after_picker.observe(refresh_after,names='value')
after_panel=widgets.VBox([widgets.HTML(value='<b>After contour: roles and Gemma disagreements</b>'),after_picker,after_output])
_combined_after_widgets=[after_picker,after_output,after_panel]
display(after_panel);refresh_after()


def inspect_group(group_id):
    group=next(g for g in groups if g['box_id']==group_id)
    page=group['page_file'];index=next(i for i,g in enumerate(analyses[page]['rows']) if g['box_id']==group_id)
    debug=debug_pages[page][index];chosen=debug['chosen']
    if chosen is None:
        print(group_id,'|',group['text_role'],'|',group['reason'])
    else:
        result=chosen['result'];region=chosen['region']
        fig,axes=plt.subplots(1,3,figsize=(12,4))
        axes[0].imshow(result['crop']);axes[0].set_title(group_id+' | '+group['text_role'])
        axes[1].imshow(region['mask'],cmap='gray');axes[1].set_title('Surrounding region')
        axes[2].imshow(result['barrier'],cmap='gray');axes[2].set_title('Faint-line barriers')
        for ax in axes:ax.axis('off')
        buffer=BytesIO()
        try:fig.savefig(buffer,format='png',bbox_inches='tight',dpi=100)
        finally:plt.close(fig)
        display(DisplayImage(data=buffer.getvalue()))
    display(combined.loc[combined['box_id'].isin(group['box_ids']),
        ['box_id','effective_text','gemma_label','text_role','individual_outline','visual_support','contour_reason']])



## 7. Export roles and deletion audit
Save all original boxes with their eligibility/role audit, remaining concatenated groups, individual outline/merge diagnostics, separate deleted-artifact and retained-box CSVs, and run settings. Source artifact results remain unchanged.

In [68]:
combined.to_csv(OUTPUT/'box_comparison.csv',index=False)
(OUTPUT/'box_comparison.json').write_text(json.dumps(comparison_rows,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
pd.DataFrame(groups).to_csv(OUTPUT/'group_predictions.csv',index=False)
(OUTPUT/'group_predictions.json').write_text(json.dumps(groups,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(OUTPUT/'grouping_audit.json').write_text(json.dumps({p:a['grouping'] for p,a in analyses.items()},indent=2),encoding='utf-8')
if len(combined):
    combined.loc[combined['artifact_stage_label']=='Artifact'].to_csv(OUTPUT/'deleted_artifacts.csv',index=False)
    combined.loc[combined['artifact_stage_label']!='Artifact'].to_csv(OUTPUT/'retained_boxes.csv',index=False)
(OUTPUT/'stage1_punctuation_audit.json').write_text(json.dumps(stage1_audit,indent=2),encoding='utf-8')
settings=dict(artifact_results=str(ARTIFACT_RESULTS),artifact_sha256=file_hash(ARTIFACT_RESULTS),
    sequence_id=SEQUENCE_ID,split=SPLIT,pages=[dict(path=str(p),sha256=file_hash(p)) for p in PAGE_FILES],
    bubble_weights_sha256=weights_hash,bubble_settings=BUBBLE_SETTINGS,
    contour_helper_sha256=file_hash(CONTOUR_HELPER),grouping_helper_sha256=file_hash(CONTOUR_HELPER.with_name('contour_grouping.py')),
    check=contour_model.CHECK,region=contour_model.REGION,faint_region=contour_model.FAINT_REGION,
    stage1='Gemma artifacts removed; empty Paddle + Magi punctuation requires bubble mask',gemma_labels_preserved=True,automatic_rescue=False,artifacts_removed_before_contour=True,tail_force_spoken=TAIL_FORCE_SPOKEN,
    tail_qualification='explicit_magi_text_tail_association',tail_sources=tail_sources)
(OUTPUT/'metadata.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
print('Saved:',OUTPUT)


Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_artifact_contour_combined\test\seq_14469aeb38963085\sequence
